In [1]:
#!pip uninstall tft_pytorch -y

In [2]:
#!pip install -U --no-cache git+https://github.com/rsscml/tft_pytorch

In [3]:
#!pip install -U openpyxl

In [4]:
# base imports
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import time
import copy
import os
import random
import warnings
from joblib import Parallel, delayed

# import TFT
import torch
from tft_pytorch import (
    OptimizedTFTDataset, 
    create_tft_dataloader, 
    create_uniform_embedding_dims,
    TemporalFusionTransformer,
    TFTTrainer,
    TFTInferenceWithTracking)

%matplotlib inline
pd.set_option("display.max_columns", 1000)
pd.set_option("display.max_rows", 1000)
warnings.filterwarnings("ignore")

from disaggregate_forecast_general import DisaggConfig, disaggregate_forecast

In [5]:
DISAGG_DATA_PATH = "../data/plant_mat_channel_agg_transitions_input.xlsx"
AGG_DATA_PATH = "../data/plant_mattransgrp_channel_agg_input.xlsx"

In [6]:
base_df = pd.read_excel(DISAGG_DATA_PATH, dtype={"Plant": 'str', "Material": 'str', "Channel": 'str'})

df = pd.read_excel(AGG_DATA_PATH, dtype={"Plant": 'str', "Channel": 'str'})

# additional feats
df['Month'] = df['YearMonth'].dt.month
df['Quarter'] = df['YearMonth'].dt.quarter

# power transform
df['Customer Demand Qty'] = np.sqrt(df['Customer Demand Qty'])

print(df.groupby(['Plant'])['key'].nunique())


Plant
3891    1580
3916    5667
Name: key, dtype: int64


In [7]:
df.head()

,Plant,transition_materials_group,Channel,YearMonth,Customer Demand Qty,Customer Demand Value,Working_Days,Material_Group,Material_Group_Desc,Material_Hierarchy,Material_Hierarchy_Desc,key,Month,Quarter
0,3891,"('0204802915',)",10,2022-01-01,0.0,0.0,19,PG016,Transmission,16ED,Add-on Gear Actuatio,"3891_('0204802915',)_10",1,1
1,3891,"('0204802915',)",10,2022-02-01,0.0,0.0,16,PG016,Transmission,16ED,Add-on Gear Actuatio,"3891_('0204802915',)_10",2,1
2,3891,"('0204802915',)",10,2022-03-01,0.0,0.0,23,PG016,Transmission,16ED,Add-on Gear Actuatio,"3891_('0204802915',)_10",3,1
3,3891,"('0204802915',)",10,2022-04-01,0.0,0.0,19,PG016,Transmission,16ED,Add-on Gear Actuatio,"3891_('0204802915',)_10",4,2
4,3891,"('0204802915',)",10,2022-05-01,0.0,0.0,19,PG016,Transmission,16ED,Add-on Gear Actuatio,"3891_('0204802915',)_10",5,2


In [8]:
# BG Plant

df = df[df['Plant']=='3891']

df.shape

(87864, 14)

In [9]:

# features config
features_config = {
    "entity_col": "key",
    "time_index_col": "YearMonth",
    "target_col": "Customer Demand Qty",

    # known in the future
    "temporal_known_numeric_col_list": ["Working_Days"],
    "temporal_known_categorical_col_list": ["Month", "Quarter"],

    # only historical
    "temporal_unknown_numeric_col_list": [],
    "temporal_unknown_categorical_col_list": [],

    # static per entity
    "static_numeric_col_list": [],
    "static_categorical_col_list": ["Channel","Material_Group","Material_Hierarchy"]
}

# context window length
historical_steps=24
forecast_steps=3
train_min_historical_steps=12
test_min_historical_steps=12
infer_min_historical_steps=0
test_periods=6

# no. of samples
train_stride=1
test_stride=1

# parallel processing
train_n_jobs = 4
test_n_jobs = 4
infer_n_jobs = 1

# scaler and encoder location (overwritten for each eval run)
scaler_path='./artefacts/train_scalers.pkl'
scaling_strategy='entity_level'
scaling_method='mean'
encoders_path='./artefacts'

# train cutoff points for various eval runs
ORIGINS = [
    (pd.Timestamp("2024-07-01"), pd.Timestamp("2025-01-01"), pd.Timestamp("2025-04-01")), # -> target April-2025
    (pd.Timestamp("2024-08-01"), pd.Timestamp("2025-02-01"), pd.Timestamp("2025-05-01")), # -> target May-2025
    (pd.Timestamp("2024-09-01"), pd.Timestamp("2025-03-01"), pd.Timestamp("2025-06-01")), # -> target Jun-2025
    (pd.Timestamp("2024-10-01"), pd.Timestamp("2025-04-01"), pd.Timestamp("2025-07-01")), # -> target Jul-2025
    (pd.Timestamp("2024-11-01"), pd.Timestamp("2025-05-01"), pd.Timestamp("2025-08-01")), # -> target Aug-2025
    (pd.Timestamp("2024-12-01"), pd.Timestamp("2025-06-01"), pd.Timestamp("2025-09-01")), # -> target Sep-2025
    (pd.Timestamp("2025-01-01"), pd.Timestamp("2025-07-01"), pd.Timestamp("2025-10-01")), # -> target Oct-2025
    (pd.Timestamp("2025-02-01"), pd.Timestamp("2025-08-01"), pd.Timestamp("2025-11-01")), # -> target Nov-2025
    (pd.Timestamp("2025-03-01"), pd.Timestamp("2025-09-01"), pd.Timestamp("2025-12-01")), # -> target Dec-2025
]


In [10]:
# Eval loop

disagg_lag3_eval_df_list = []
lag3_eval_df_list = []

for i, origin in enumerate(ORIGINS):

    print(f"Eval run: {i+1}")
    
    # ----------------- split dataframe ------------------------
    train_start = pd.to_datetime('2020-01-01', format="%Y-%m-%d")
    train_cutoff = origin[0]
    
    test_start = train_cutoff - pd.DateOffset(months=historical_steps)
    test_cutoff = origin[1]

    infer_cutoff = origin[2]
    infer_start = infer_cutoff - pd.DateOffset(months=historical_steps + forecast_steps)

    print(f" train_start: {train_start}, train_cutoff: {train_cutoff}")
    print(f" test_start: {test_start}, test_cutoff: {test_cutoff}")
    print(f" infer_start: {infer_start}, infer_cutoff: {infer_cutoff}")
    
    train_df = df[df['YearMonth'] <= train_cutoff].copy()
    test_df = df[(df['YearMonth'] > test_start) & (df['YearMonth'] <= test_cutoff)].copy()
    
    infer_df = df[df['YearMonth'] <= infer_cutoff].copy()
    infer_df =  infer_df.groupby(['key'], group_keys=False).tail(historical_steps + forecast_steps)
    
    print(train_df.shape, test_df.shape, infer_df.shape)

    # ------------------ create datasets ----------------------------
    train_dataset = OptimizedTFTDataset(
                        data_source=train_df,
                        features_config=features_config, 
                        historical_steps=historical_steps,
                        prediction_steps=forecast_steps,
                        stride=train_stride,
                        enable_padding=True,
                        padding_strategy = 'zero',
                        categorical_padding_value = -1,
                        min_historical_steps=train_min_historical_steps,
                        allow_inference_only_entities=False,
                        scaler_path=scaler_path,
                        scaling_strategy=scaling_strategy,
                        scaling_method=scaling_method,
                        cold_start_scaler_cols=None,
                        mean_scaler_epsilon=1.0,
                        recency_alpha=1.0,
                        n_jobs=train_n_jobs,
                        max_series=None,
                        mode='train',
                        encoders_path=encoders_path,
                        fit_encoders=None,
                        preprocessing_fn=None)
    
    test_dataset = OptimizedTFTDataset(
                        data_source=test_df,
                        features_config=features_config, 
                        historical_steps=historical_steps,
                        prediction_steps=forecast_steps,
                        stride=test_stride,
                        enable_padding=True,
                        padding_strategy = 'zero',
                        categorical_padding_value = -1,
                        min_historical_steps=test_min_historical_steps,
                        allow_inference_only_entities=False,
                        scaler_path=scaler_path,
                        scaling_strategy=scaling_strategy,
                        scaling_method=scaling_method,
                        cold_start_scaler_cols=None,
                        mean_scaler_epsilon=1.0,
                        recency_alpha=0,
                        n_jobs=test_n_jobs,
                        max_series=None,
                        mode='test',
                        encoders_path=encoders_path,
                        fit_encoders=None,
                        preprocessing_fn=None)

    infer_dataset = OptimizedTFTDataset(
                        data_source=infer_df,
                        features_config=features_config, 
                        historical_steps=historical_steps,
                        prediction_steps=forecast_steps,
                        stride=test_stride,
                        enable_padding=True,
                        padding_strategy = 'zero',
                        categorical_padding_value = -1,
                        min_historical_steps=infer_min_historical_steps,
                        allow_inference_only_entities=True,
                        scaler_path=scaler_path,
                        scaling_strategy=scaling_strategy,
                        scaling_method=scaling_method,
                        cold_start_scaler_cols=['Material_Group','Channel'], 
                        mean_scaler_epsilon=1.0,
                        recency_alpha=0,
                        n_jobs=1,
                        max_series=None,
                        mode='test',
                        encoders_path=encoders_path,
                        fit_encoders=None,
                        preprocessing_fn=None)

    # ------------------------ create dataloaders -------------------------------
    train_dataloader, train_adapter = create_tft_dataloader(train_dataset, batch_size=64, shuffle=True, num_workers=0, drop_last=False, pin_memory=True)

    test_dataloader, test_adapter = create_tft_dataloader(test_dataset, batch_size=64, shuffle=False, num_workers=0, drop_last=False, pin_memory=True)

    infer_dataloader, infer_adapter = create_tft_dataloader(infer_dataset, batch_size=64, shuffle=False, num_workers=0, drop_last=False, pin_memory=True)

    # ------------------- create embedding dims dict for various categorical variables
    categorical_embedding_dims = create_uniform_embedding_dims(train_dataset, hidden_layer_size=160)
    num_static_categorical=len(train_dataset.static_categorical_cols)
    num_static_continuous=len(train_dataset.static_numeric_cols)
    num_historical_categorical=len(train_dataset.temporal_unknown_categorical_cols) + len(train_dataset.temporal_known_categorical_cols)
    num_historical_continuous=len(train_dataset.target_cols) + len(train_dataset.temporal_unknown_numeric_cols) + len(train_dataset.temporal_known_numeric_cols)
    num_future_categorical=len(train_dataset.temporal_known_categorical_cols)
    num_future_continuous=len(train_dataset.temporal_known_numeric_cols)

    # ---------------------- create model -----------------------------------
    model = TemporalFusionTransformer(
            # Model architecture parameters
            hidden_layer_size = 160,
            num_attention_heads = 1,
            num_lstm_layers = 1,
            num_attention_layers = 4,
            dropout_rate = 0.1,
            
            # Input specification
            num_static_categorical = num_static_categorical,
            num_static_continuous = num_static_continuous,
            num_historical_categorical = num_historical_categorical,
            num_historical_continuous = num_historical_continuous,
            num_future_categorical = num_future_categorical,
            num_future_continuous = num_future_continuous,
        
            # embedding dims for categorical variables
            categorical_embedding_dims = categorical_embedding_dims,
            
            # Time dimensions
            historical_steps = historical_steps,
            prediction_steps = forecast_steps,
            
            # Output specification
            num_outputs = 1,
            device = 'cuda' #'cuda' if torch.cuda.is_available() else 'cpu'
        )
    
    # ------------------- create trainer ------------------------------------
    trainer = TFTTrainer(model = model,
                     # data params
                     train_loader = train_dataloader,
                     val_loader = test_dataloader,
                     train_adapter = train_adapter,
                     val_adapter = test_adapter,
                     # loss params
                     loss_type = 'huber',
                     loss_params = {'delta': 1.0},
                     # optimizer params
                     optimizer_type = 'adam',
                     learning_rate = 0.0001,
                     # Scheduler configuration
                     scheduler_type = 'reduce_on_plateau',
                     scheduler_factor = 0.5,
                     scheduler_patience = 5,
                     # Training options
                     enable_gradient_clipping = True,
                     max_grad_norm = 1.0,
                     enable_train_sample_weighting = False,
                     enable_val_sample_weighting = False,
                     # Mixed Precision Training
                     enable_mixed_precision = False,
                     # Checkpointing
                     save_path = './models/ch_plant_mattrans_ch',
                     save_every = 1)
    
    # ------------------------ train -----------------------------------------
    trainer.train(num_epochs=50, patience=10)

    # ----------------------- infer ------------------------------------------
    inferencewithtracking = TFTInferenceWithTracking(model_path = 'models/ch_plant_mattrans_ch/best_model.pt', model = model, adapter = infer_adapter, device = 'cuda')
    results_df = inferencewithtracking.predict_with_metadata(infer_dataloader)

    # ----------------------- inverse power transform -------------------------
    results_df['pred_0'] = np.square(results_df['pred_0'])
    results_df['actual_Customer Demand Qty'] = np.square(results_df['actual_Customer Demand Qty'])
    results_df.rename(columns={'entity_id':'key', 'timestamp': 'YearMonth'}, inplace=True)

    # get lag3
    lag3_results_df = results_df[results_df['YearMonth']==origin[2]]
    
    # merge with base columns
    infer_df = infer_df[['key','Plant','transition_materials_group','Channel']]
    infer_df.drop_duplicates(inplace=True)
    lag3_results_df = lag3_results_df.merge(infer_df, on=['key'], how='left')
    
    # write out
    lag3_results_df.to_csv(f"./outputs/ch_hub1_transform_plant_mattrans_channel_{str(origin[2])}.csv")
    
    lag3_eval_df_list.append(lag3_results_df)

    # disaggregate to Material
    cfg = DisaggConfig(
        group_key_cols=["Plant", "transition_materials_group", "Channel"], # key columns in training dataset
        item_col="Material", # disaggregation level (Plant, Material, Channel)
        time_col="YearMonth",
        historical_qty_col="Customer Demand Qty",
        forecast_qty_col="pred_0",
        output_qty_col="Disaggregated Forecast Qty",
        proportion_col="proportion",
        output_key_col="key",
        output_key_parts=["Plant", "Material", "Channel"],
        output_key_sep="_",
    )

    disagg_lag3_results_df = disaggregate_forecast(
        forecast_df=lag3_results_df,
        historical_df=base_df,
        config=cfg,
        cutoff_date=origin[1] + pd.DateOffset(months=1), # first month of forecast
        lookback_months=6,
    )

    disagg_lag3_eval_df_list.append(disagg_lag3_results_df)

    # ----------------------- clear model ------------------------------------- 
    del model
    gc.collect()



print("Combine & save results")
results_df = pd.concat(lag3_eval_df_list, ignore_index=True)
disagg_results_df = pd.concat(disagg_lag3_eval_df_list, ignore_index=True)

# merge actuals at disagg level for comparison
base_df = base_df[['key','YearMonth','Customer Demand Qty']].drop_duplicates()
disagg_results_df = disagg_results_df.merge(base_df, on=['key','YearMonth'], how='left')

# Save
results_df.to_csv("./outputs/ch_hub1_transform_plant_mattrans_channel_lag3eval.csv", index=False)
disagg_results_df.to_csv("./outputs/ch_hub1_disagg_plant_material_channel_lag3eval_mattrans.csv", index=False)


Eval run: 1
 train_start: 2020-01-01 00:00:00, train_cutoff: 2024-07-01 00:00:00
 test_start: 2022-07-01 00:00:00, test_cutoff: 2025-01-01 00:00:00
 infer_start: 2023-01-01 00:00:00, infer_cutoff: 2025-04-01 00:00:00
(63469, 14) (43050, 14) (42225, 14)

Feature Configuration:
  Targets: 1
  Static: 0 numeric, 3 categorical
  Known temporal: 1 numeric, 2 categorical
  Unknown temporal: 0 numeric, 0 categorical
  Total numeric features: 2
Will save scalers to ./artefacts/train_scalers.pkl after fitting
Loading data...
Processing 1580 entities...
  Using parallel processing with 'threading' backend...
Loaded 1580 valid series from 1580 total
Computing padding values for each entity...
Computed padding values for 1580 entities
Fitting encoders on train data...
  Fitting encoder for Channel...
    -> 4 unique values
  Fitting encoder for Material_Group...
    -> 14 unique values
  Fitting encoder for Material_Hierarchy...
    -> 146 unique values
  Fitting encoder for Month...
    -> 12 uni

2026-03-26 09:06:31,418 - INFO - Starting training for 50 epochs...
2026-03-26 09:06:31,418 - INFO - 
Epoch 1/50
2026-03-26 09:06:31,861 - INFO - Batch 0/357, Loss: 0.5129
2026-03-26 09:06:35,325 - INFO - Batch 100/357, Loss: 0.2704
2026-03-26 09:06:38,665 - INFO - Batch 200/357, Loss: 0.2679
2026-03-26 09:06:41,950 - INFO - Batch 300/357, Loss: 0.2145
2026-03-26 09:06:43,863 - INFO - Train Loss: 0.2645
2026-03-26 09:06:44,970 - INFO - Val Loss: 0.4089, Metrics: {'mse': 4.61023473739624, 'rmse': 2.14714571871502, 'mae': 0.5571550130844116, 'mape': nan, 'val_loss': 0.4089491528445958}
2026-03-26 09:06:44,970 - INFO - New best validation loss: 0.4089
2026-03-26 09:06:45,100 - INFO - Saved checkpoint to models/ch_plant_mattrans_ch/checkpoint_epoch_1.pt
2026-03-26 09:06:45,206 - INFO - Saved best model to models/ch_plant_mattrans_ch/best_model.pt
2026-03-26 09:06:45,287 - INFO - 
Epoch 2/50
2026-03-26 09:06:45,327 - INFO - Batch 0/357, Loss: 0.2656
2026-03-26 09:06:48,684 - INFO - Batch 10

Computed proportions for 5449 item-within-group combinations.

Aggregated forecast total : 104,745.87
Disaggregated forecast total: 104,675.48
Difference (leakage)       : 70.38
Eval run: 2
 train_start: 2020-01-01 00:00:00, train_cutoff: 2024-08-01 00:00:00
 test_start: 2022-08-01 00:00:00, test_cutoff: 2025-02-01 00:00:00
 infer_start: 2023-02-01 00:00:00, infer_cutoff: 2025-05-01 00:00:00
(64904, 14) (43050, 14) (42225, 14)

Feature Configuration:
  Targets: 1
  Static: 0 numeric, 3 categorical
  Known temporal: 1 numeric, 2 categorical
  Unknown temporal: 0 numeric, 0 categorical
  Total numeric features: 2
Will save scalers to ./artefacts/train_scalers.pkl after fitting
Loading data...
Processing 1580 entities...
  Using parallel processing with 'threading' backend...
Loaded 1580 valid series from 1580 total
Computing padding values for each entity...
Computed padding values for 1580 entities
Fitting encoders on train data...
  Fitting encoder for Channel...
    -> 4 unique values

2026-03-26 09:14:14,393 - INFO - Starting training for 50 epochs...
2026-03-26 09:14:14,394 - INFO - 
Epoch 1/50
2026-03-26 09:14:14,479 - INFO - Batch 0/380, Loss: 0.1659



=== Encoding Summary ===
Static Categorical Encodings (first 3 entities):
  Entity 3891_('0204802915',)_10: Channel=0, Material_Group=3, Material_Hierarchy=24
  Entity 3891_('0204802917',)_10: Channel=0, Material_Group=3, Material_Hierarchy=22
  Entity 3891_('0204802919',)_10: Channel=0, Material_Group=3, Material_Hierarchy=25

Temporal Categorical Encodings (first entity, first 10 timesteps):
  Month: [ 2  3  4  5  6  7  8  9 10 11]
  Quarter: [0 1 1 1 2 2 2 3 3 3]
Data preprocessing complete!

Dataset Statistics
Series: 1580
Windows: 1,580
Padded windows: 145
Features: 2 numeric, 5 categorical

Memory Usage:
  Scaler parameters: 0.0 MB
  Array shape: (1580, 2)


...

Computed proportions for 5449 item-within-group combinations.

Aggregated forecast total : 95,474.71
Disaggregated forecast total: 95,309.54
Difference (leakage)       : 165.17
Eval run: 9
 train_start: 2020-01-01 00:00:00, train_cutoff: 2025-03-01 00:00:00
 test_start: 2023-03-01 00:00:00, test_cutoff: 2025-09-01 00:00:00
 infer_start: 2023-09-01 00:00:00, infer_cutoff: 2025-12-01 00:00:00
(74949, 14) (43050, 14) (42225, 14)

Feature Configuration:
  Targets: 1
  Static: 0 numeric, 3 categorical
  Known temporal: 1 numeric, 2 categorical
  Unknown temporal: 0 numeric, 0 categorical
  Total numeric features: 2
Will save scalers to ./artefacts/train_scalers.pkl after fitting
Loading data...
Processing 1580 entities...
  Using parallel processing with 'threading' backend...
Loaded 1580 valid series from 1580 total
Computing padding values for each entity...
Computed padding values for 1580 entities
Fitting encoders on train data...
  Fitting encoder for Channel...
    -> 4 unique values


2026-03-26 10:07:13,703 - INFO - Starting training for 50 epochs...
2026-03-26 10:07:13,704 - INFO - 
Epoch 1/50
2026-03-26 10:07:13,802 - INFO - Batch 0/536, Loss: 0.8843



=== Encoding Summary ===
Static Categorical Encodings (first 3 entities):
  Entity 3891_('0204802915',)_10: Channel=0, Material_Group=3, Material_Hierarchy=24
  Entity 3891_('0204802917',)_10: Channel=0, Material_Group=3, Material_Hierarchy=22
  Entity 3891_('0204802919',)_10: Channel=0, Material_Group=3, Material_Hierarchy=25

Temporal Categorical Encodings (first entity, first 10 timesteps):
  Month: [ 9 10 11  0  1  2  3  4  5  6]
  Quarter: [3 3 3 0 0 0 1 1 1 2]
Data preprocessing complete!

Dataset Statistics
Series: 1580
Windows: 1,580
Padded windows: 145
Features: 2 numeric, 5 categorical

Memory Usage:
  Scaler parameters: 0.0 MB
  Array shape: (1580, 2)


2026-03-26 10:07:17,283 - INFO - Batch 100/536, Loss: 0.1872
2026-03-26 10:07:20,855 - INFO - Batch 200/536, Loss: 0.4724
2026-03-26 10:07:24,296 - INFO - Batch 300/536, Loss: 0.1120
2026-03-26 10:07:27,706 - INFO - Batch 400/536, Loss: 0.3037
2026-03-26 10:07:31,181 - INFO - Batch 500/536, Loss: 0.9106
2026-03-26 10:07:32,387 - INFO - Train Loss: 0.3105
2026-03-26 10:07:33,473 - INFO - Val Loss: 0.8471, Metrics: {'mse': 16.311769485473633, 'rmse': 4.038783168910363, 'mae': 1.001912236213684, 'mape': nan, 'val_loss': 0.8471016943092562}
2026-03-26 10:07:33,474 - INFO - New best validation loss: 0.8471
2026-03-26 10:07:33,598 - INFO - Saved checkpoint to models/ch_plant_mattrans_ch/checkpoint_epoch_1.pt
2026-03-26 10:07:33,693 - INFO - Saved best model to models/ch_plant_mattrans_ch/best_model.pt
2026-03-26 10:07:33,758 - INFO - 
Epoch 2/50
2026-03-26 10:07:33,799 - INFO - Batch 0/536, Loss: 0.2541
2026-03-26 10:07:37,248 - INFO - Batch 100/536, Loss: 0.2952
2026-03-26 10:07:40,594 - IN

Computed proportions for 5449 item-within-group combinations.

Aggregated forecast total : 89,048.64
Disaggregated forecast total: 88,852.72
Difference (leakage)       : 195.92
Combine & save results
